# 17 — Bài báo, bước 2: chấm điểm chung năm chiến lược (máy local)

**Phạm vi**

Lần chấm duy nhất cho các mô hình mới của bước 2 trên Hold-out, focused set và Lazada, theo amendment v3.2 (mục 5), v3.3, v3.4
và v3.5: TF-IDF (D), PhoBERT (E), flat Bi-LSTM và flat LLM, đặt cạnh ba chiến lược của GĐ10. Năm chiến lược xếp theo họ:
TF-IDF, Bi-LSTM + fastText, ViSoBERT, PhoBERT, LLM few-shot.

**Cấu hình được chấm**

Năm cascade, năm mô hình phẳng bốn lớp và các hybrid (tầng 1 của X, tầng 2 của Y). Hybrid thiếu dự đoán tầng 2 cho review mà
tầng 1 giữ lại thì bỏ qua và liệt kê, như v3.1.

**Ràng buộc**

Chỉ dùng dự đoán đã lưu và cấu hình đã khóa; không train lại, không gọi API, không đổi ngưỡng hay prompt. Trước khi chấm, notebook
kiểm tra file khóa và tái tạo đúng các bảng GĐ10; lệch bất kỳ ô nào thì dừng.

In [1]:
import os, sys, json, datetime, subprocess
import numpy as np, pandas as pd, yaml
ROOT = os.path.abspath('..')
sys.path.insert(0, os.path.join(ROOT, '02_src'))
from src.utils.seeds import set_seed
set_seed(42)
from src.classification.final_eval import load_frozen
from src.classification.evaluate import FOUR_CLASSES
from src.classification import paper_analysis as PA
from src.classification.strategy_llm_fewshot import prompts as PR
from src.classification.strategy_llm_fewshot.classify import cost_summary
from src.utils.llm import get_settings, prices
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30); pd.set_option('display.max_rows', 200)
DATA = os.path.join(ROOT, '01_data', 'processed')
RES = os.path.join(ROOT, '05_experiments', 'rq1_classification')
RUN_DATE = datetime.date.today().strftime('%d/%m/%Y')
COMMIT = subprocess.run(['git', '-C', ROOT, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
FZ = load_frozen(ROOT)

def frozen_yaml(fname, key, must):
    txt = open(os.path.join(ROOT, '03_configs', 'frozen', fname), encoding='utf-8').read()
    assert txt.startswith('# FROZEN'), f'{fname}: thiếu header FROZEN'
    for m in must:
        assert m in txt, f'{fname}: thiếu dấu "{m}"'
    return yaml.safe_load(txt)[key]

TF = frozen_yaml('paper_tfidf_config.yaml', 'tfidf', ['amendment v3.2'])
NC = frozen_yaml('paper_neural_config.yaml', 'paper_neural', ['amendment v3.4', 'amendment v3.5'])
assert NC['phobert']['tier1']['threshold'] == 0.02 and NC['phobert']['segmenter'] == 'vncorenlp-rdrsegmenter'
SETS = {'Train': 'shopee_train.csv', 'Validation': 'shopee_validation.csv', 'Hold-out': 'shopee_holdout.csv',
        'Focused': 'shopee_focused_feature_ux.csv', 'Lazada': 'lazada_external_test.csv'}
D = {k: pd.read_csv(os.path.join(DATA, f), encoding='utf-8-sig') for k, f in SETS.items()}
FLAT_HASH = PR.load_prompt('flat', 'v1', D['Train']).hash(get_settings())
assert FLAT_HASH in open(PR.REGISTRY, encoding='utf-8').read(), 'prompt phẳng khác bản đã khóa'
print('Ngày chạy:', RUN_DATE, '| commit:', COMMIT)
print('Ngưỡng tầng 1 đã khóa — TF-IDF:', TF['tier1']['threshold'], '| PhoBERT:', NC['phobert']['tier1']['threshold'],
      '| prompt phẳng', FLAT_HASH[:12], 'khớp registry')

Ngày chạy: 15/11/2026 | commit: 9e19ef75c14d7dd0ebc9aa95cda542a8238a303d
Ngưỡng tầng 1 đã khóa — TF-IDF: 0.45 | PhoBERT: 0.02 | prompt phẳng 62baaf4f076a khớp registry


## Dự đoán và tái tạo GĐ10

**Dựng cấu hình**

Dự đoán của từng chiến lược được dựng từ file đã lưu với ngưỡng đã khóa, rồi ghép thành cascade, phẳng và hybrid.

**Đối chiếu**

Các bảng tầng 1, tầng 2 và end-to-end của ba chiến lược cũ phải trùng từng ô với file `gd10_*`.

In [2]:
LBL = {l: i for i, l in enumerate(FOUR_CLASSES)}
NAME = {'holdout': 'Hold-out', 'focused': 'Focused', 'lazada': 'Lazada'}
IDS = {s: D[n]['review_id'].tolist() for s, n in NAME.items()}
Y4 = {s: D[n]['gold_label'].map(LBL).to_numpy() for s, n in NAME.items()}
PRED = {s: PA.step2_predictions(RES, s, IDS[s], FZ, TF, NC) for s in NAME}
CFG, SKIP = {}, {}
for s in ('holdout', 'lazada'):
    CFG[s], SKIP[s] = PA.configurations_step2(*PRED[s])
    print(f"{NAME[s]}: {len(CFG[s])} cấu hình | hybrid bỏ qua: {[(d['tầng 1'], d['tầng 2'], d['review thiếu tầng 2']) for d in SKIP[s]]}")

def check(new, ref_file, key, cols, rename=None):
    ref = pd.read_csv(os.path.join(RES, ref_file)).set_index(key)[cols].astype(str)
    got = new.assign(**{key: new[key].replace(rename or {})}).set_index(key).loc[ref.index, cols].astype(str)
    bad = got != ref
    assert not bad.any().any(), f'{ref_file}: lệch\n{pd.concat([got[bad.any(axis=1)], ref[bad.any(axis=1)]], axis=1)}'
    print(f'{ref_file}: khớp {len(ref)} dòng × {len(cols)} cột')

T1H = PA.tier1_scores(Y4['holdout'], PRED['holdout'][0])
check(T1H[0], 'gd10_tier1_holdout.csv', 'chiến lược', ['F1 ACT', 'Recall ACT', 'Precision ACT', 'Macro-F1 (2 lớp)'])
T2H = PA.tier2_scores(Y4['holdout'], PRED['holdout'][1])
check(T2H[0], 'gd10_tier2_holdout.csv', 'chiến lược', ['Macro-F1', 'F1 BUG', 'F1 FEATURE', 'F1 UX'])
E2EH = PA.e2e_scores(Y4['holdout'], CFG['holdout'])
check(E2EH[0], 'gd10_e2e_holdout.csv', 'cấu hình', ['Macro-F1 (4 lớp)', 'F1 BUG', 'F1 FEATURE', 'F1 UX', 'F1 OTHER'],
      rename={'Phẳng ViSoBERT': 'Baseline phẳng ViSoBERT'})
E2EL = PA.e2e_scores(Y4['lazada'], CFG['lazada'])
check(E2EL[0], 'gd10_e2e_lazada.csv', 'cấu hình', ['Macro-F1 (4 lớp)', 'F1 BUG', 'F1 FEATURE', 'F1 UX', 'F1 OTHER'])

Hold-out: 28 cấu hình | hybrid bỏ qua: [('TF-IDF', 'LLM few-shot', 17), ('PhoBERT', 'LLM few-shot', 28)]
Lazada: 28 cấu hình | hybrid bỏ qua: [('TF-IDF', 'LLM few-shot', 15), ('PhoBERT', 'LLM few-shot', 26)]


gd10_tier1_holdout.csv: khớp 3 dòng × 4 cột
gd10_tier2_holdout.csv: khớp 3 dòng × 4 cột


gd10_e2e_holdout.csv: khớp 4 dòng × 5 cột


gd10_e2e_lazada.csv: khớp 3 dòng × 5 cột


## Tầng 1 trên Hold-out

**Chỉ số**

F1, recall và precision của ACTIONABLE ở ngưỡng đã khóa, Macro-F1 hai lớp; neural là trung bình ± độ lệch chuẩn qua ba seed. Kiểm
định McNemar giữa năm chiến lược, hiệu chỉnh Holm trong họ tầng 1.

**Độ nhạy ngưỡng PhoBERT**

PhoBERT tầng 1 được chấm thêm ở 0,01 (đầu ra của luật v2.0 trước v3.5) và 0,5 (argmax), kèm cascade tương ứng; ngưỡng chính là 0,02.

In [3]:
cols1 = ['chiến lược', 'F1 ACT', 'Recall ACT', 'Precision ACT', 'Macro-F1 (2 lớp)', 'f1 CI', 'recall CI', 'số lần chạy']
print('Tầng 1 — Hold-out (n = 1.000; %):'); print(T1H[0][cols1].to_string(index=False))
MC = {'tầng 1 – Hold-out': PA.mcnemar_table(T1H[1], T1H[2], 'tầng 1 – Hold-out')}
SENS = []
for thr in (0.01, NC['phobert']['tier1']['threshold'], 0.5):
    t1s, t2s, _ = PA.step2_predictions(RES, 'holdout', IDS['holdout'], FZ, TF, NC, phobert_threshold=thr)
    tab = PA.tier1_scores(Y4['holdout'], {'PhoBERT': t1s['PhoBERT']})[0].iloc[0]
    cas, _ = PA.configurations_step2({'PhoBERT': t1s['PhoBERT']}, {'PhoBERT': t2s['PhoBERT']}, {}, hybrids=False)
    e = PA.e2e_scores(Y4['holdout'], cas)[0].iloc[0]
    SENS.append({'ngưỡng PhoBERT tầng 1': thr, 'vai trò': 'chính (v3.5)' if thr == NC['phobert']['tier1']['threshold'] else 'độ nhạy',
                 '% gọi ACTIONABLE': ' / '.join(f'{a.mean() * 100:.0f}' for a in t1s['PhoBERT'].values()),
                 'F1 ACT': tab['F1 ACT'], 'Recall ACT': tab['Recall ACT'], 'Precision ACT': tab['Precision ACT'],
                 'Macro-F1 (2 lớp)': tab['Macro-F1 (2 lớp)'], 'Cascade Macro-F1 (4 lớp)': e['Macro-F1 (4 lớp)']})
SENS = pd.DataFrame(SENS)
print('\nĐộ nhạy ngưỡng PhoBERT tầng 1 — Hold-out (%):'); print(SENS.to_string(index=False))

Tầng 1 — Hold-out (n = 1.000; %):
        chiến lược     F1 ACT Recall ACT Precision ACT Macro-F1 (2 lớp)     f1 CI recall CI  số lần chạy
            TF-IDF 68.1 ± 0.0 78.7 ± 0.0    60.0 ± 0.0       79.8 ± 0.0 62.7–73.0 72.7–84.4            3
Bi-LSTM + fastText 59.7 ± 4.7 84.0 ± 7.4    47.0 ± 7.5       72.6 ± 5.0 57.1–67.0 77.4–88.4            3
          ViSoBERT 65.9 ± 0.9 69.8 ± 4.0    62.7 ± 2.9       78.9 ± 0.6 59.2–70.2 61.0–74.4            3
           PhoBERT 64.0 ± 5.1 81.1 ± 4.1    53.5 ± 8.3       76.4 ± 4.4 60.3–70.8 72.0–83.9            3
      LLM few-shot       81.7       86.9          77.2             88.7 77.4–85.6 81.8–91.6            1



Độ nhạy ngưỡng PhoBERT tầng 1 — Hold-out (%):
 ngưỡng PhoBERT tầng 1      vai trò % gọi ACTIONABLE     F1 ACT Recall ACT Precision ACT Macro-F1 (2 lớp) Cascade Macro-F1 (4 lớp)
                  0.01      độ nhạy    71 / 51 / 100 40.3 ± 9.1 97.4 ± 4.0    25.8 ± 7.6      41.1 ± 23.4              34.3 ± 15.6
                  0.02 chính (v3.5)     25 / 24 / 36 64.0 ± 5.1 81.1 ± 4.1    53.5 ± 8.3       76.4 ± 4.4               60.8 ± 2.7
                  0.50      độ nhạy     14 / 15 / 16 70.6 ± 1.5 64.1 ± 1.6    78.6 ± 2.9       82.4 ± 0.9               66.4 ± 1.2


## Tầng 2 trên Hold-out và focused set

**Chỉ số**

Macro-F1 ba lớp và F1 từng lớp trên review gold-actionable, tách khỏi lỗi của tầng 1. Focused set là tập làm giàu FEATURE/UX,
báo cáo riêng.

In [4]:
cols2 = ['chiến lược', 'n', 'Macro-F1', 'F1 BUG', 'F1 FEATURE', 'F1 UX', 'Macro-F1 CI', 'số lần chạy']
print('Tầng 2 — Hold-out, gold-actionable (%):'); print(T2H[0][cols2].to_string(index=False))
MC['tầng 2 – Hold-out'] = PA.mcnemar_table(T2H[1], T2H[2], 'tầng 2 – Hold-out')
FOC = PA.tier2_scores(Y4['focused'], PRED['focused'][1])
print('\nTầng 2 — focused set, gold-actionable (%):'); print(FOC[0][cols2].to_string(index=False))

Tầng 2 — Hold-out, gold-actionable (%):
        chiến lược   n   Macro-F1     F1 BUG F1 FEATURE      F1 UX Macro-F1 CI  số lần chạy
            TF-IDF 183 74.2 ± 0.0 87.3 ± 0.0 49.1 ± 0.0 86.4 ± 0.0   66.5–81.0            3
Bi-LSTM + fastText 183 69.6 ± 1.1 79.6 ± 0.5 50.5 ± 3.5 78.8 ± 1.5   61.6–77.6            3
          ViSoBERT 183 76.5 ± 1.6 85.6 ± 0.9 57.8 ± 3.3 86.0 ± 0.6   69.5–83.8            3
           PhoBERT 183 78.2 ± 0.3 85.8 ± 0.5 62.1 ± 1.2 86.9 ± 0.9   69.9–85.2            3
      LLM few-shot 183       89.7       94.0       83.3       91.9   83.8–94.5            1

Tầng 2 — focused set, gold-actionable (%):
        chiến lược  n   Macro-F1      F1 BUG F1 FEATURE      F1 UX Macro-F1 CI  số lần chạy
            TF-IDF 91 62.6 ± 0.0  23.5 ± 0.0 85.2 ± 0.0 79.1 ± 0.0   51.6–73.5            3
Bi-LSTM + fastText 91 68.8 ± 3.5 42.9 ± 10.2 85.4 ± 1.4 78.1 ± 1.8   60.2–82.9            3
          ViSoBERT 91 76.2 ± 3.2 57.7 ± 11.0 91.0 ± 1.5 80.0 ± 0.0   67.5–88.3          

## End-to-end bốn lớp trên Hold-out

**Chỉ số**

Macro-F1 bốn lớp và F1 từng lớp (trung bình ± độ lệch chuẩn qua các lần chạy), Macro-F1 của lần chạy chuẩn với khoảng tin cậy 95%,
và recall, precision của ACTIONABLE khi gộp BUG, FEATURE, UX. Kiểm định McNemar cho mọi cặp cấu hình, hiệu chỉnh Holm trong họ.

In [5]:
ACT_H = PA.actionable_scores(Y4['holdout'], CFG['holdout'])
def e2e_view(tab, act):
    v = tab.merge(act, on='cấu hình')
    v['CI 95%'] = [f'{a * 100:.1f}–{b * 100:.1f}' for a, b in zip(v['CI 2,5%'], v['CI 97,5%'])]
    v['kiến trúc'] = v['cấu hình'].map(PA.architecture)
    v[['Recall ACT', 'Precision ACT']] = (v[['Recall ACT', 'Precision ACT']] * 100).round(1)
    return v[['kiến trúc', 'cấu hình', 'Macro-F1 (4 lớp)', 'CI 95%', 'F1 BUG', 'F1 FEATURE', 'F1 UX', 'F1 OTHER', 'Recall ACT', 'Precision ACT', 'số lần chạy']]
VIEW_H = e2e_view(E2EH[0], ACT_H)
print('End-to-end — Hold-out (n = 1.000; %):'); print(VIEW_H.to_string(index=False))
MC['end-to-end – Hold-out'] = PA.mcnemar_table(E2EH[2], E2EH[3], 'end-to-end – Hold-out')
m = MC['end-to-end – Hold-out']
print(f"\nMcNemar end-to-end Hold-out: {int(m['khác biệt (α = 0,05)'].sum())}/{len(m)} cặp khác biệt có ý nghĩa sau Holm")

End-to-end — Hold-out (n = 1.000; %):
kiến trúc                          cấu hình Macro-F1 (4 lớp)    CI 95%     F1 BUG F1 FEATURE      F1 UX   F1 OTHER  Recall ACT  Precision ACT  số lần chạy
  cascade                    Cascade TF-IDF       61.3 ± 0.0 56.9–65.5 53.2 ± 0.0 21.3 ± 0.0 79.4 ± 0.0 91.4 ± 0.0        78.7           60.0            3
  cascade        Cascade Bi-LSTM + fastText       53.5 ± 3.7 49.9–58.3 40.2 ± 2.1 22.2 ± 7.3 66.1 ± 3.6 85.5 ± 5.2        84.0           47.0            3
  cascade                  Cascade ViSoBERT       62.1 ± 0.7 57.6–67.8 48.5 ± 2.0 34.8 ± 3.6 73.2 ± 0.4 91.8 ± 0.5        69.8           62.7            3
  cascade                   Cascade PhoBERT       60.8 ± 2.7 56.5–66.4 44.9 ± 5.4 31.8 ± 1.8 77.5 ± 2.0 88.8 ± 3.7        81.1           53.5            3
  cascade              Cascade LLM few-shot             81.3 76.3–85.5       71.9       69.1       88.5       95.6        86.9           77.2            1
    phẳng                      P


McNemar end-to-end Hold-out: 193/378 cặp khác biệt có ý nghĩa sau Holm


## Cascade và phẳng trong từng họ mô hình

**Nội dung**

Với mỗi chiến lược, cascade được đặt cạnh mô hình phẳng của chính nó: Macro-F1 trung bình, chênh của lần chạy chuẩn với khoảng tin
cậy bootstrap ghép cặp, McNemar (Holm trong họ end-to-end) và recall ACTIONABLE. Đây là phép so sánh trực tiếp lợi ích của kiến trúc
hai tầng.

In [6]:
ARCH_H = PA.architecture_contrast(E2EH[0], E2EH[1], MC['end-to-end – Hold-out'], ACT_H)
print('Cascade − phẳng — Hold-out (%):'); print(ARCH_H.round({c: 1 for c in ARCH_H.columns if c != 'McNemar p (Holm)'}).round({'McNemar p (Holm)': 4}).to_string(index=False))

Cascade − phẳng — Hold-out (%):
        họ mô hình  Macro-F1 cascade  Macro-F1 phẳng  chênh cascade − phẳng (lần chạy chuẩn)  CI 2,5%  CI 97,5%  McNemar p (Holm)  Recall ACT cascade  Recall ACT phẳng
            TF-IDF              61.3            62.8                                    -1.4     -4.8       2.1            0.5262                78.7              65.6
Bi-LSTM + fastText              53.5            59.7                                    -8.7    -14.2      -2.8            0.0000                84.0              75.2
          ViSoBERT              62.1            66.8                                    -3.8     -8.6       1.2            0.0008                69.8              59.6
           PhoBERT              60.8            70.9                                    -9.1    -14.5      -3.6            0.0000                81.1              66.3
      LLM few-shot              81.3            80.0                                     1.3     -2.0       4.6            1.000

## Lazada

**Nội dung**

End-to-end của mọi cấu hình trên Lazada với mô hình và prompt đã khóa, mức suy giảm so với Hold-out, so sánh cascade và phẳng, và
McNemar trong họ Lazada.

In [7]:
ACT_L = PA.actionable_scores(Y4['lazada'], CFG['lazada'])
VIEW_L = e2e_view(E2EL[0], ACT_L)
print('End-to-end — Lazada (n = 500; %):'); print(VIEW_L.to_string(index=False))
MC['end-to-end – Lazada'] = PA.mcnemar_table(E2EL[2], E2EL[3], 'end-to-end – Lazada')
ARCH_L = PA.architecture_contrast(E2EL[0], E2EL[1], MC['end-to-end – Lazada'], ACT_L)
print('\nCascade − phẳng — Lazada (%):'); print(ARCH_L.round(1).to_string(index=False))
DROP = E2EH[0][['cấu hình', 'Macro-F1 (trung bình)']].merge(E2EL[0][['cấu hình', 'Macro-F1 (trung bình)']], on='cấu hình', suffixes=(' Hold-out', ' Lazada'))
DROP['chênh (điểm %)'] = (DROP['Macro-F1 (trung bình) Lazada'] - DROP['Macro-F1 (trung bình) Hold-out']) * 100
DROP[['Macro-F1 (trung bình) Hold-out', 'Macro-F1 (trung bình) Lazada']] *= 100
print('\nSuy giảm Hold-out → Lazada (%):'); print(DROP.round(1).to_string(index=False))

End-to-end — Lazada (n = 500; %):
kiến trúc                          cấu hình Macro-F1 (4 lớp)    CI 95%     F1 BUG F1 FEATURE      F1 UX   F1 OTHER  Recall ACT  Precision ACT  số lần chạy
  cascade                    Cascade TF-IDF       56.6 ± 0.0 50.2–62.5 41.8 ± 0.0 26.7 ± 0.0 72.7 ± 0.0 85.4 ± 0.0        88.9           41.9            3
  cascade        Cascade Bi-LSTM + fastText       47.1 ± 6.4 43.0–55.1 31.9 ± 5.5 21.3 ± 6.7 55.3 ± 5.9 80.0 ± 8.3        91.8           37.1            3
  cascade                  Cascade ViSoBERT       56.0 ± 0.4 48.6–62.5 42.4 ± 1.6 33.8 ± 2.1 59.0 ± 2.4 88.8 ± 1.3        81.1           47.9            3
  cascade                   Cascade PhoBERT       52.7 ± 3.6 48.8–62.5 37.6 ± 5.5 28.4 ± 6.3 63.3 ± 4.4 81.6 ± 7.2        88.9           38.2            3
  cascade              Cascade LLM few-shot             60.5 52.1–67.7       48.1       37.5       66.7       89.8        79.0           50.0            1
    phẳng                      Phẳng


Cascade − phẳng — Lazada (%):
        họ mô hình  Macro-F1 cascade  Macro-F1 phẳng  chênh cascade − phẳng (lần chạy chuẩn)  CI 2,5%  CI 97,5%  McNemar p (Holm)  Recall ACT cascade  Recall ACT phẳng
            TF-IDF              56.6            56.1                                     0.5     -4.2       5.7               0.0                88.9              70.4
Bi-LSTM + fastText              47.1            50.7                                    -5.3    -12.2       1.6               0.0                91.8              85.6
          ViSoBERT              56.0            65.3                                    -9.7    -14.8      -4.5               0.0                81.1              66.3
           PhoBERT              52.7            67.4                                   -11.1    -17.5      -4.5               0.0                88.9              74.9
      LLM few-shot              60.5            59.5                                     1.0     -3.4       6.0               1.0

## Hold-out khử trùng

**Nội dung**

Như v3.1: bỏ 217 review Hold-out có văn bản chuẩn hóa trùng Train ∪ Validation, chấm lại mọi cấu hình. Đây là phân tích độ bền.

In [8]:
REF_TXT = pd.concat([D['Train']['text'], D['Validation']['text']])
KEEP = {s: PA.keep_mask(D[NAME[s]]['text'], REF_TXT) for s in ('holdout', 'lazada')}
E2EHD = PA.e2e_scores(Y4['holdout'], CFG['holdout'], KEEP['holdout'])
DED = E2EH[0][['cấu hình', 'Macro-F1 (trung bình)']].merge(E2EHD[0][['cấu hình', 'Macro-F1 (trung bình)']], on='cấu hình', suffixes=(' đầy đủ', ' khử trùng'))
DED['chênh (điểm %)'] = (DED['Macro-F1 (trung bình) khử trùng'] - DED['Macro-F1 (trung bình) đầy đủ']) * 100
DED[['Macro-F1 (trung bình) đầy đủ', 'Macro-F1 (trung bình) khử trùng']] *= 100
print(f"Hold-out khử trùng: {KEEP['holdout'].sum()} review"); print(DED.round(1).to_string(index=False))
rk = lambda col: DED[col].rank(ascending=False, method='min')
print('Tương quan hạng Spearman giữa hai bảng xếp hạng:', round(float(DED[['Macro-F1 (trung bình) đầy đủ', 'Macro-F1 (trung bình) khử trùng']].corr('spearman').iloc[0, 1]), 3))

Hold-out khử trùng: 783 review
                         cấu hình  Macro-F1 (trung bình) đầy đủ  Macro-F1 (trung bình) khử trùng  chênh (điểm %)
                   Cascade TF-IDF                          61.3                             60.4            -1.0
       Cascade Bi-LSTM + fastText                          53.5                             51.9            -1.6
                 Cascade ViSoBERT                          62.1                             61.2            -0.9
                  Cascade PhoBERT                          60.8                             59.5            -1.3
             Cascade LLM few-shot                          81.3                             80.9            -0.4
                     Phẳng TF-IDF                          62.8                             62.3            -0.5
         Phẳng Bi-LSTM + fastText                          59.7                             58.6            -1.0
                   Phẳng ViSoBERT                          66.8  

## Phân tích lỗi

**Phân rã lỗi**

Bỏ sót actionable, giữ nhầm OTHER và sai loại trong actionable, trung bình qua các lần chạy, trên Hold-out và Lazada.

**Theo lớp**

Precision, recall và F1 từng lớp của các cấu hình thuần (năm cascade và năm phẳng) trên Hold-out, cùng ma trận nhầm lẫn của mọi cấu hình.

In [9]:
ERR_H, ERR_L = PA.error_decomposition(Y4['holdout'], CFG['holdout']), PA.error_decomposition(Y4['lazada'], CFG['lazada'])
print('Phân rã lỗi — Hold-out:'); print(ERR_H.to_string(index=False))
print('\nPhân rã lỗi — Lazada:'); print(ERR_L.to_string(index=False))
PURE = {k: v for k, v in CFG['holdout'].items() if PA.architecture(k) != 'hybrid'}
PC_H = PA.per_class(Y4['holdout'], PURE)
CONF_H = PA.confusion_long(Y4['holdout'], CFG['holdout'])
print('\nRecall từng lớp — cấu hình thuần, Hold-out (%):')
print(PC_H.pivot(index='cấu hình', columns='lớp', values='recall')[FOUR_CLASSES].loc[list(PURE)].to_string())

Phân rã lỗi — Hold-out:
                         cấu hình    n  tổng lỗi  bỏ sót actionable  giữ nhầm OTHER  sai loại  % bỏ sót actionable  % giữ nhầm OTHER  % sai loại
                   Cascade TF-IDF 1000     163.0               39.0            96.0      28.0                 23.9              58.9        17.2
       Cascade Bi-LSTM + fastText 1000     250.0               29.3           182.7      38.0                 11.7              73.1        15.2
                 Cascade ViSoBERT 1000     151.0               55.3            76.7      19.0                 36.6              50.8        12.6
                  Cascade PhoBERT 1000     195.7               34.7           135.3      25.7                 17.7              69.2        13.1
             Cascade LLM few-shot 1000      80.0               24.0            47.0       9.0                 30.0              58.8        11.2
                     Phẳng TF-IDF 1000     134.0               63.0            55.0      16.0             

## Chi phí và biên Pareto

**Nguồn chi phí**

Thời gian suy luận cho 1.000 review và chi phí API đo ở GĐ10 (ViSoBERT, Bi-LSTM tầng 1 và 2, ViSoBERT phẳng trên GPU T4; LLM qua API),
notebook 14 (TF-IDF, CPU local), notebook 16 (PhoBERT và Bi-LSTM phẳng trên GPU T4, PhoBERT cộng thời gian tách từ) và notebook 15
(flat LLM). Cascade và hybrid tính tầng 1 trên mọi review cộng tầng 2 trên phần tầng 1 giữ lại (tỷ lệ giữ của lần chạy chuẩn trên Hold-out).

**Biên Pareto**

Tính riêng theo chi phí API và theo thời gian suy luận, với Macro-F1 end-to-end của lần chạy chuẩn trên Hold-out. Phần cứng khác nhau
giữa các nguồn nên thời gian chỉ so được theo bậc độ lớn.

**Hình**

Chỉ ghi nhãn các điểm trên biên (và flat LLM ở panel chi phí API, để thấy nó đắt hơn cascade LLM); vị trí của mọi cấu hình nằm trong `paper_step2_pareto.csv`.

In [10]:
MODEL = {'visobert': 'ViSoBERT', 'bilstm': 'Bi-LSTM + fastText', 'tfidf': 'TF-IDF', 'phobert': 'PhoBERT'}
SEC, USD, TRAIN, HW = {}, {}, {}, {}
for f in ('gd10_costs_neural.csv', 'paper_tfidf_costs.csv', 'paper_neural_costs.csv'):
    for _, r in pd.read_csv(os.path.join(RES, f)).iterrows():
        k = (r['tier'], MODEL[r['model']])
        SEC[k], USD[k], TRAIN[k], HW[k] = r['infer_seconds_per_1000_mean'], 0.0, r['train_seconds_mean'], r['gpu']
CL = pd.read_csv(os.path.join(RES, 'gd10_llm_costs.csv')).set_index('bước')
for tier, step in (('tier1', 'holdout – tầng 1'), ('tier2', 'holdout – tầng 2')):
    k = (tier, PA.LLM)
    SEC[k], USD[k], TRAIN[k], HW[k] = CL.loc[step, 'giây_/1000_review'], CL.loc[step, 'usd_/1000_review'], 0.0, 'API'
cs = cost_summary(pd.read_csv(os.path.join(RES, 'paper_flat_llm_holdout.csv')), prices(), 'flat LLM')
SEC[('flat', PA.LLM)], USD[('flat', PA.LLM)], TRAIN[('flat', PA.LLM)], HW[('flat', PA.LLM)] = cs['giây_/1000_review'], cs['usd_/1000_review'], 0.0, 'API'
SEG = {'PhoBERT': float(pd.read_csv(os.path.join(RES, 'paper_neural_costs.csv')).query("model == 'phobert'")['segment_seconds_per_1000'].iloc[0])}
KEEPRATE = {x: float(PA.canonical(PRED['holdout'][0][x]).mean()) for x in PA.ORDER}
LABELS_NEEDED = {x: 'Train 1.200 + Validation 350' for x in PA.ORDER}
LABELS_NEEDED[PA.LLM] = '14–17 ví dụ Train + Validation 350'
rows = []
for name in CFG['holdout']:
    s, u, t = PA.config_cost(name, KEEPRATE, SEC, USD, TRAIN, SEG)
    parts = [name[len(PA.FLAT_PREFIX):]] if PA.architecture(name) == 'phẳng' else (
        [name[len('Cascade '):]] if name.startswith('Cascade ') else name.split(' → '))
    rows.append({'cấu hình': name, 'mã': PA.short_label(name), 'kiến trúc': PA.architecture(name),
                 'giây / 1.000 review': s, 'USD API / 1.000 review': u, 'giây train / lần chạy': t,
                 'nhãn cần': ' ; '.join(sorted({LABELS_NEEDED[p] for p in parts})),
                 'gọi API': any(p == PA.LLM for p in parts)})
COST = pd.DataFrame(rows)
PAR = E2EH[0][['cấu hình', 'Macro-F1 chuẩn', 'CI 2,5%', 'CI 97,5%']].rename(columns={'Macro-F1 chuẩn': 'Macro-F1'}).merge(COST, on='cấu hình')
PAR['Pareto (USD)'] = PA.pareto_front(PAR['Macro-F1'], PAR['USD API / 1.000 review'])
PAR['Pareto (giây)'] = PA.pareto_front(PAR['Macro-F1'], PAR['giây / 1.000 review'])
PAR['thuần'] = PAR['kiến trúc'] != 'hybrid'
PAR['nhãn USD'], PAR['nhãn giây'] = PAR['cấu hình'] == PA.FLAT_PREFIX + PA.LLM, False
show = PAR.sort_values(['USD API / 1.000 review', 'giây / 1.000 review']).copy()
show[['Macro-F1', 'CI 2,5%', 'CI 97,5%']] = (show[['Macro-F1', 'CI 2,5%', 'CI 97,5%']] * 100).round(1)
print(show[['mã', 'cấu hình', 'Macro-F1', 'CI 2,5%', 'CI 97,5%', 'giây / 1.000 review', 'USD API / 1.000 review', 'giây train / lần chạy',
            'Pareto (USD)', 'Pareto (giây)']].round(4).to_string(index=False))
TXT = {'vi': {'y': 'Macro-F1 end-to-end 4 lớp (%)', 'cascade': 'Cascade thuần', 'phẳng': 'Phẳng 4 lớp', 'hybrid': 'Hybrid (tầng 1 → tầng 2)',
              'front': 'Biên Pareto', 'dominated': 'Bị trội', 'flat_code': 'V phẳng',
              'x': ['Chi phí API (USD / 1.000 review)', 'Thời gian suy luận (giây / 1.000 review, thang log)']},
       'en': {'y': 'End-to-end Macro-F1, 4 classes (%)', 'cascade': 'Pure cascade', 'phẳng': 'Flat 4-class', 'hybrid': 'Hybrid (tier 1 → tier 2)',
              'front': 'Pareto front', 'dominated': 'Dominated', 'flat_code': 'V flat',
              'x': ['API cost (USD per 1,000 reviews)', 'Inference time (s per 1,000 reviews, log scale)']}}
for lang, t in TXT.items():
    panels = [('USD API / 1.000 review', t['x'][0], False, 'Pareto (USD)'), ('giây / 1.000 review', t['x'][1], True, 'Pareto (giây)')]
    PA.pareto_figure(PAR, os.path.join(RES, f'paper_step2_pareto_{lang}.png'), panels, t, label_col=['nhãn USD', 'nhãn giây'])
print('Hình: paper_step2_pareto_vi.png, paper_step2_pareto_en.png (nhãn cho điểm trên biên và flat LLM ở panel API)')

     mã                          cấu hình  Macro-F1  CI 2,5%  CI 97,5%  giây / 1.000 review  USD API / 1.000 review  giây train / lần chạy  Pareto (USD)  Pareto (giây)
    T→B       TF-IDF → Bi-LSTM + fastText      58.9     54.0      63.4               0.0381                  0.0000                 1.4106         False           True
B phẳng          Phẳng Bi-LSTM + fastText      62.9     56.4      68.7               0.0533                  0.0000                 2.3735         False           True
    T→T                    Cascade TF-IDF      61.3     56.9      65.5               0.0589                  0.0000                 0.3754         False          False
    B→B        Cascade Bi-LSTM + fastText      54.2     49.9      58.3               0.1090                  0.0000                 6.3728         False          False
T phẳng                      Phẳng TF-IDF      62.8     57.5      67.6               0.1131                  0.0000                 0.6362         False        

Hình: paper_step2_pareto_vi.png, paper_step2_pareto_en.png (nhãn cho điểm trên biên và flat LLM ở panel API)


## Lưu kết quả

**Các file**

Mọi bảng được lưu với tiền tố `paper_step2_` trong `05_experiments/rq1_classification/`, cùng `paper_step2_run_info.json`.

In [11]:
def save(df, name):
    df.to_csv(os.path.join(RES, f'paper_step2_{name}.csv'), index=False, encoding='utf-8-sig')

save(T1H[0], 'tier1_holdout'); save(SENS, 'phobert_threshold_sensitivity'); save(T2H[0], 'tier2_holdout'); save(FOC[0], 'tier2_focused')
save(VIEW_H, 'e2e_holdout'); save(E2EH[0], 'e2e_holdout_full'); save(E2EH[1], 'e2e_holdout_diff')
save(VIEW_L, 'e2e_lazada'); save(E2EL[0], 'e2e_lazada_full'); save(E2EL[1], 'e2e_lazada_diff'); save(DROP, 'holdout_vs_lazada')
save(ARCH_H, 'architecture_holdout'); save(ARCH_L, 'architecture_lazada')
save(pd.concat(MC.values(), ignore_index=True), 'mcnemar')
save(DED, 'dedup_holdout'); save(ERR_H, 'errors_holdout'); save(ERR_L, 'errors_lazada'); save(PC_H, 'per_class_holdout'); save(CONF_H, 'confusion_holdout')
save(COST, 'costs'); save(PAR, 'pareto')
json.dump({'run_date': RUN_DATE, 'commit': COMMIT, 'python': sys.version.split()[0], 'amendments': ['v3.2', 'v3.3', 'v3.4', 'v3.5'],
           'n_boot': 10_000, 'bootstrap_seed': 42, 'skipped_hybrids': SKIP, 'flat_prompt_hash': FLAT_HASH,
           'phobert_threshold': NC['phobert']['tier1']['threshold'], 'tfidf_threshold': TF['tier1']['threshold']},
          open(os.path.join(RES, 'paper_step2_run_info.json'), 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print('Đã lưu:', sorted(f for f in os.listdir(RES) if f.startswith('paper_step2_')))

Đã lưu: ['paper_step2_architecture_holdout.csv', 'paper_step2_architecture_lazada.csv', 'paper_step2_confusion_holdout.csv', 'paper_step2_costs.csv', 'paper_step2_dedup_holdout.csv', 'paper_step2_e2e_holdout.csv', 'paper_step2_e2e_holdout_diff.csv', 'paper_step2_e2e_holdout_full.csv', 'paper_step2_e2e_lazada.csv', 'paper_step2_e2e_lazada_diff.csv', 'paper_step2_e2e_lazada_full.csv', 'paper_step2_errors_holdout.csv', 'paper_step2_errors_lazada.csv', 'paper_step2_holdout_vs_lazada.csv', 'paper_step2_mcnemar.csv', 'paper_step2_pareto.csv', 'paper_step2_pareto_en.png', 'paper_step2_pareto_vi.png', 'paper_step2_per_class_holdout.csv', 'paper_step2_phobert_threshold_sensitivity.csv', 'paper_step2_run_info.json', 'paper_step2_tier1_holdout.csv', 'paper_step2_tier2_focused.csv', 'paper_step2_tier2_holdout.csv']


## Nhận xét

**Tính hợp lệ của lần chấm**

Các bảng GĐ10 được tái tạo khớp từng ô; ngưỡng tầng 1 dùng đúng bản khóa (TF-IDF 0,45, PhoBERT 0,02 theo v3.5) và prompt phẳng khớp
registry. Hai hybrid TF-IDF → LLM và PhoBERT → LLM bị bỏ vì LLM tầng 2 thiếu dự đoán cho 17 và 28 review Hold-out mà tầng 1 giữ lại.

**Tầng 1**

LLM dẫn đầu với F1 ACTIONABLE 81,7. Trong các mô hình có huấn luyện, TF-IDF đạt F1 ACTIONABLE cao nhất (68,1), tiếp theo ViSoBERT
(65,9 ± 0,9), PhoBERT (64,0 ± 5,1) và Bi-LSTM + fastText (59,7 ± 4,7); khoảng tin cậy của bốn mô hình này chồng lấn nhau.

**Tầng 2**

LLM đạt Macro-F1 89,7, PhoBERT 78,2 ± 0,3, ViSoBERT 76,5 ± 1,6, TF-IDF 74,2 và Bi-LSTM 69,6 ± 1,1. FEATURE vẫn là lớp khó nhất với các
mô hình có huấn luyện (F1 49,1–62,1).

**End-to-end**

Cascade LLM (81,3) và flat LLM (80,0) dẫn đầu. Trong các mô hình có huấn luyện, PhoBERT phẳng cao nhất (70,9 ± 1,0), tiếp theo ViSoBERT
phẳng (66,8 ± 1,5). Hybrid tốt nhất là LLM → PhoBERT (74,6).

**Cascade và phẳng**

Ở bốn họ có huấn luyện, mô hình phẳng có Macro-F1 cao hơn cascade của chính nó: chênh cascade − phẳng là −1,4 (TF-IDF), −8,7 (Bi-LSTM),
−3,8 (ViSoBERT) và −9,1 (PhoBERT), khoảng tin cậy không chứa 0 ở Bi-LSTM và PhoBERT. Ngược lại, cascade có recall ACTIONABLE cao hơn ở
cả bốn họ (78,7 so với 65,6; 84,0 so với 75,2; 69,8 so với 59,6; 81,1 so với 66,3). Với LLM, hai kiến trúc tương đương (chênh +1,3,
khoảng tin cậy −2,0 đến +4,6) và bản phẳng có recall cao hơn (90,7 so với 86,9).

**Lazada**

PhoBERT phẳng (67,4 ± 2,1) và ViSoBERT phẳng (65,3 ± 2,5) cao nhất, chỉ giảm 3,5 và 1,5 điểm so với Hold-out. LLM giảm khoảng 20 điểm
ở cả hai kiến trúc (cascade 81,3 → 60,5; phẳng 80,0 → 59,5), nên mức suy giảm gắn với LLM và prompt chứ không với kiến trúc. Cascade
thua phẳng rõ nhất ở ViSoBERT (−9,7) và PhoBERT (−11,1), chủ yếu vì giữ nhầm OTHER (69,5% và 82,1% số lỗi).

**Độ bền**

Trên 783 review Hold-out khử trùng, mọi cấu hình giảm 0,4–1,6 điểm và tương quan hạng Spearman giữa hai bảng xếp hạng là 0,996.

**Độ nhạy ngưỡng PhoBERT**

Ở ngưỡng 0,01, cascade PhoBERT chỉ đạt 34,3 ± 15,6 vì seed 2024 gọi mọi review là ACTIONABLE. Ở 0,5, cascade đạt 66,4 ± 1,2 nhưng
recall ACTIONABLE còn 64,1. Ngưỡng chính 0,02 cho 60,8 ± 2,7 với recall 81,1.

**Chi phí**

Theo chi phí API, biên Pareto gồm PhoBERT phẳng (70,8; 0 USD), LLM → PhoBERT (74,6; 0,173 USD cho 1.000 review) và cascade LLM (81,3;
0,206 USD). Flat LLM (80,0) tốn 0,266 USD nên bị cascade LLM trội về chi phí API, nhưng nhanh hơn (791 so với 1.103 giây cho 1.000
review). Các cấu hình không gọi API xử lý 1.000 review trong 0,04–2,4 giây; TF-IDF đo trên CPU, mô hình neural trên GPU T4, nên thời
gian chỉ so được theo bậc độ lớn.

**Giới hạn**

LLM chỉ chạy một lần với temperature 0; Hold-out chỉ có 24 review FEATURE nên kết luận về FEATURE cần đọc cùng khoảng tin cậy và focused set.